In [1]:
from utils import *
from FAPT import *
from DRAG import *
import matplotlib.pyplot as plt
 
simulation = Simulation()
dim_q1, dim_q2, dim_c = (3,3,3)

# Steuerungs-Flag für symbolische oder numerische Belegung
symbolic = False
tg_val = 250.0  # Gate-Zeit [ns]
sigma_r_g_val = 0.3
sigma_r_t_val = 0.1

In [2]:
import time
import numpy as np
import sympy as sp
from IPython.display import Math, display


print(
    f"[Block 1] Initialisiere Parameter, Energien & hermitesches V1"
    f" (symbolic={symbolic})..."
)
start_time = time.perf_counter()

# 1.1 Zeit- und Puls-Symbole
t_sym = sp.Symbol("t", real=True, positive=True)
tg_sym = sp.Symbol("t_g", real=True, positive=True)
sigma_sym = sp.Symbol(r"\sigma", real=True, positive=True)

# 1.2 Frequenz- und Steuer-Symbole
wd0_sym = sp.Symbol(r"\omega_{d0}", real=True)
dwd_t_sym = sp.Function(r"\delta\omega_d", real=True)(t_sym)
wd_t_sym = wd0_sym + dwd_t_sym

# Komplexes A(t) = A_r(t) + i A_i(t)
Ar_sym = sp.Function("A_r", real=True)(t_sym)
Ai_sym = sp.Function("A_i", real=True)(t_sym)
A_t_sym = Ar_sym + sp.I * Ai_sym

# Freie physikalische System-Parameter
Delta_sym = sp.Symbol("Delta", real=True)
lam_sym = sp.Symbol("lambda", real=True)

# 2. Dynamic State/Matrix Init (Symbolisch vs. Numerisch)
D = 27
V0_sym = sp.zeros(D, D)

if symbolic:
  # --- SYMBOLISCHER PFAD ---
  E_array_sym = np.array([sp.Symbol(f"E_{i}", real=True) for i in range(D)])

  V1_dressed_ref = simulation.V1_dressed_array
  threshold = 1e-8
  V1_sym = np.zeros((D, D), dtype=object)

  for i1 in range(dim_q1):
    for i2 in range(dim_q2):
      for ic in range(dim_c):
        for jc in range(dim_c):
          for j1 in range(dim_q1):
            for j2 in range(dim_q2):
              i = i1 * dim_c * dim_q2 + ic * dim_q2 + i2
              j = j1 * dim_c * dim_q2 + jc * dim_q2 + j2

              if np.abs(V1_dressed_ref[i, j]) >= threshold:
                real = i1 == j1 and ic == jc and i2 == j2
                real = True
                bra_ket = sp.Symbol(
                    f"\\bra{{{i1}{ic}{i2}}}\\widetilde{{V_1}}{{\\ket{{{j1}{jc}{j2}}}}}",
                    real=real,
                )
                V1_sym[i, j] = bra_ket
              else:
                V1_sym[i, j] = sp.S.Zero
                  # Physikalische Parameter
else:
  # --- NUMERISCHER PFAD ---
  E_array_sym = np.array(simulation.E_array, dtype=object)
  V1_sym = np.array(simulation.V1_dressed_array, dtype=object)


[Block 1] Initialisiere Parameter, Energien & hermitesches V1 (symbolic=False)...


In [3]:
print("[Block 2] Konstruiere gefilterte symbolische Matrix V1 (V0 = 0) und starte FAPT...")
start_time = time.perf_counter()


# 2.3 FAPT Heff symbolisch aufrufen
print("  ➜ Berechne Heff via FAPT analytisch...", end="", flush=True)
heff_start = time.perf_counter()

Heff_sym = Heff_Floquet_total_matrix_summed(
    rH=getattr(simulation, 'rH', 1),
    wd=wd_t_sym,
    A=A_t_sym,
    resonances=simulation.resonances,
    E=E_array_sym,                  # REIN SYMBOLISCHE ENERGIEN
    V_posHarm=V1_sym,                # GEFILTERTE SYMBOLISCHE BRA-KET MATRIX
    V0=V0_sym,                       # EXPLIZIT SP.ZEROS
    ref_state=getattr(simulation, 'ref_state', None),
    dwd=0,
    dA=0,
    t=t_sym,
    analytics=True,
    include_geometric=False,
    include_micromotion=False,
    include_g_correction=False,
    verbose=False,
    rW=getattr(simulation, 'rW', 1)
)

print(f" Fertig in {time.perf_counter() - heff_start:.2f}s!")

# ==============================================================================
# SYMBOLISCHE KONDENSATION / LINEARISIERUNG
# ==============================================================================
def condense_element_symbolic(expr, var, order=1, factor_vars=None):
    if expr == 0 or expr == sp.S.Zero:
        return sp.S.Zero
    if factor_vars is None:
        factor_vars = []

    terms = sp.Add.make_args(expr)
    linearized_terms = []

    for term in terms:
        if not term.has(var):
            linearized_terms.append(sp.cancel(term))
            continue

        f_0 = term.subs(var, 0)
        if order == 0:
            linearized_terms.append(sp.cancel(f_0))
        elif order == 1:
            df_0 = term.diff(var).subs(var, 0)
            linearized_terms.append(sp.cancel(f_0) + sp.cancel(df_0) * var)

    total_sum = sp.Add(*linearized_terms)
    poly_dwd = sp.Poly(total_sum, var)
    coeffs = poly_dwd.all_coeffs()
    degree = poly_dwd.degree()

    compact_terms = []
    for idx, coeff in enumerate(coeffs):
        current_pow = degree - idx
        coeff_factored = sp.factor(sp.collect(coeff, factor_vars))
        if current_pow == 0:
            compact_terms.append(coeff_factored)
        else:
            compact_terms.append(coeff_factored * (var**current_pow))

    return sp.Add(*compact_terms)

# Linearisierung der symbolischen Matrix
H_eff_raw = sp.Matrix(Heff_sym)
delta_shift = H_eff_raw[0, 0]
H_eff_shifted = H_eff_raw - delta_shift * sp.eye(H_eff_raw.shape[0])

rows, cols = H_eff_shifted.rows, H_eff_shifted.cols
Heff = sp.zeros(rows, cols)

for i in range(rows):
    for j in range(cols):
        elem = H_eff_shifted[i, j]
        if elem != 0:
            Heff[i, j] = condense_element_symbolic(elem, dwd_t_sym, order=1, factor_vars=[Ar_sym, Ai_sym])

if symbolic:
    Delta_val = -1.2
    lambda_val = 10.2
else:
    Delta_sol = -2*Heff[1,1] + Heff[2,2]
    lambda_val = Heff[1,2] / Heff[0,1]

print(f"✅ Block 2 abgeschlossen in {time.perf_counter() - start_time:.2f}s\n")
display(Math(rf"(H_{{\text{{eff}}}})_{{0,1}} = {sp.latex(Heff[0, 1])}"))
display(Math(rf"(H_{{\text{{eff}}}})_{{1,2}} = {sp.latex(Heff[1,2])}"))

[Block 2] Konstruiere gefilterte symbolische Matrix V1 (V0 = 0) und starte FAPT...
  ➜ Berechne Heff via FAPT analytisch... Fertig in 0.98s!
✅ Block 2 abgeschlossen in 1.04s



<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [4]:
print("[Block 3] Berechne H_target via DRAG mit symbolischen Zeitparametern...")
start_time = time.perf_counter()

H_target_raw, subs_ht = h_target_symbolic_abstract(
    Delta=Delta_sym,
    rrr=lam_sym, 
    tg=tg_sym,               # SYMBOLISCHES tg
    sigma_r=sigma_sym, # SYMBOLISCHES sigma (sigma_r = sigma / tg)
    pulse='gauss', 
    t=t_sym, 
    order=4
)

H_target = sp.Matrix(H_target_raw).doit()

print(f"✅ Block 3 abgeschlossen in {time.perf_counter() - start_time:.2f}s\n")

[Block 3] Berechne H_target via DRAG mit symbolischen Zeitparametern...
✅ Block 3 abgeschlossen in 0.35s



In [5]:
import time
import sympy as sp
from IPython.display import display, Math

print("[Block 4] Extrahiere die 3 Bestimmungsgleichungen aus H_eff und H_target...")
start_time = time.perf_counter()

# 1. Differenzmatrix bilden
H_diff = Heff - H_target

# 3. Dynamische Extraktion aus den Matrixelementen und Ersetzung anwenden
raw_re_01 = sp.re(H_diff[0, 1])
raw_im_01 = sp.im(H_diff[0, 1])
raw_diag_11 = sp.re(H_diff[1, 1])

# Gleichungen mit explizitem "= 0"
equations = [
    sp.Eq(raw_re_01, 0),
    sp.Eq(raw_im_01, 0),
    sp.Eq(raw_diag_11, 0)
]

print(f"✅ Block 4 abgeschlossen in {time.perf_counter() - start_time:.2f}s")
print("  -> 3 Gleichungen dynamisch extrahiert.\n")

# 4. Strings für LaTeX-Ausgabe aufbereiten
str_eq1 = sp.latex(equations[0])
str_eq2 = sp.latex(equations[1])
str_eq3 = sp.latex(equations[2])

display(Math(rf"\text{{1. }} A_r(t)\text{{-Gl. (Re 0,1)}}: \quad {str_eq1}"))
display(Math(rf"\text{{2. }} A_i(t)\text{{-Gl. (Im 0,1)}}: \quad {str_eq2}"))
display(Math(rf"\text{{3. }} \delta\omega_d(t)\text{{-Gl. (Diag 1,1)}}: \quad {str_eq3}"))

# 5. LaTeX-Code inklusive Definition der Hilfsfunktionen generieren
latex_system_code = rf"""% --- Definition der Hilfsfunktionen ---

% --- Bestimmungsgleichungen ---
\begin{{aligned}}
  \text{{1. }} A_r(t)\text{{-Gl. (Re 0,1)}} &: \quad {str_eq1} \\[2ex]
  \text{{2. }} A_i(t)\text{{-Gl. (Im 0,1)}} &: \quad {str_eq2} \\[2ex]
  \text{{3. }} \delta\omega_d(t)\text{{-Gl. (Diag 1,1)}} &: \quad {str_eq3}
\end{{aligned}}"""

# 6. Ausgabe als Roh-Text zum Kopieren
print("\n--- REINER LATEX-CODE (ZUM KOPIEREN) ---")
print(latex_system_code)

[Block 4] Extrahiere die 3 Bestimmungsgleichungen aus H_eff und H_target...
✅ Block 4 abgeschlossen in 0.02s
  -> 3 Gleichungen dynamisch extrahiert.



<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- REINER LATEX-CODE (ZUM KOPIEREN) ---
% --- Definition der Hilfsfunktionen ---

% --- Bestimmungsgleichungen ---
\begin{aligned}
  \text{1. } A_r(t)\text{-Gl. (Re 0,1)} &: \quad 0.0106205947176049 A_{r}{\left(t \right)} - \frac{\operatorname{{\mathcal E}}_\pi{\left(t \right)}}{2} = 0 \\[2ex]
  \text{2. } A_i(t)\text{-Gl. (Im 0,1)} &: \quad - 0.0106205947176049 A_{i}{\left(t \right)} - \frac{\dot{\mathcal E}_\pi}{2 \Delta} = 0 \\[2ex]
  \text{3. } \delta\omega_d(t)\text{-Gl. (Diag 1,1)} &: \quad - 4.43804328641986 \left(0.225324526027031 \omega_{d0} - 1.0\right) - 1.0 \delta\omega_{d}{\left(t \right)} - \frac{\left(\lambda^{2} - 4\right) \operatorname{{\mathcal E}}_\pi^{2}{\left(t \right)}}{4 \Delta} = 0
\end{aligned}


In [6]:
# 1. dwd aus Gl. 3 auflösen
sol_dwd = sp.solve(equations[2], dwd_t_sym)[0]

# 2. Gleichungen vorerst normal expandieren
V1_diag_list = list(V1_sym.diagonal())

eq1_expanded = equations[0].expand(complex=True)
eq2_expanded = equations[1].expand(complex=True)

# 3. A_i und A_r bestimmen
sol_Ai_from_eq2 = sp.solve(eq2_expanded, Ai_sym)[0]
sol_Ar_from_eq1 = sp.solve(eq1_expanded, Ar_sym)[0]

eq_Ar = sp.Eq(sol_Ar_from_eq1.subs(Ai_sym, sol_Ai_from_eq2), Ar_sym)
sol_Ar = sp.solve(eq_Ar, Ar_sym)[0]
sol_Ai = sol_Ai_from_eq2.subs(Ar_sym, sol_Ar)

# 5. LaTeX-Strings erstellen
str_Ar = sp.latex(sp.Eq(Ar_sym, sol_Ar))
str_Ai = sp.latex(sp.Eq(Ai_sym, sol_Ai))
str_dwd = sp.latex(sp.Eq(dwd_t_sym, sol_dwd))

# 6. Anzeige aller drei Lösungen
display(Math(rf"\text{{Hauptamplitude }} A_r(t): \quad {str_Ar}"))
display(Math(rf"\text{{DRAG-Korrektur }} A_i(t): \quad {str_Ai}"))
display(Math(rf"\text{{Frequenzkorrektur }} \delta\omega_d(t): \quad {str_dwd}"))
# 7. Reiner LaTeX-Code zum Kopieren
latex_output = rf"""\begin{{aligned}}
  A_r(t) &= {sp.latex(sol_Ar)} \\[2ex]
  A_i(t) &= {sp.latex(sol_Ai)} \\[2ex]
  \delta\omega_d(t) &= {sp.latex(sol_dwd)}
\end{{aligned}}"""

print("\n--- REINER LATEX-CODE (ZUM KOPIEREN) ---")
print(latex_output)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- REINER LATEX-CODE (ZUM KOPIEREN) ---
\begin{aligned}
  A_r(t) &= 47.0783429077837 \operatorname{{\mathcal E}}_\pi{\left(t \right)} \\[2ex]
  A_i(t) &= - \frac{47.0783429077837 \dot{\mathcal E}_\pi}{\Delta} \\[2ex]
  \delta\omega_d(t) &= \frac{4.43804328641986 \cdot 10^{-15} \Delta \left(1.0 \cdot 10^{15} - 225324526027031.0 \omega_{d0}\right) - 0.25 \lambda^{2} \operatorname{{\mathcal E}}_\pi^{2}{\left(t \right)} + \operatorname{{\mathcal E}}_\pi^{2}{\left(t \right)}}{\Delta}
\end{aligned}


In [7]:
# Determination of wd0 
ep1, ep2 = subs_ht.keys()
wd0_eq = sp.Eq(sol_dwd.subs({t_sym:0, ep1:0, ep2:0}),0)
wd0_sol = sp.solve(wd0_eq, wd0_sym)[0]
if not symbolic:
    Delta_val = Delta_sol.subs({wd0_sym: wd0_sol})
wd0_sol

4.43804328641986

In [8]:
# ==============================================================================
# BLOCK 1: SYMBOLISCHE REDUKTION AUF t, sigma UND tg
# ==============================================================================

import re
import time
import numpy as np
import sympy as sp


# ------------------------------------------------------------------------------
# 1. Feste numerische Parameter
# ------------------------------------------------------------------------------

DELTA_VALUE = -1.18
LAMBDA_VALUE = 10.0
COUPLING_VALUE = 1.0 + 0.01j
DEFAULT_NUMERIC_VALUE = 1.0
EXPANSION_ORDER = 4

ALLOWED_SYMBOLS = {t_sym, sigma_sym, tg_sym}


# ------------------------------------------------------------------------------
# 2. Hilfsfunktionen
# ------------------------------------------------------------------------------

def collect_free_symbols(expressions):
    return set().union(*(sp.sympify(expression).free_symbols for expression in expressions))


def find_wd0_symbol(expression):
    symbols = expression.free_symbols

    preferred = [
        symbol
        for symbol in symbols
        if any(name in re.sub(r"[^a-z0-9]", "", symbol.name.lower()) for name in ("wd0", "omegad0"))
    ]

    if preferred:
        return preferred[0]

    fallback = [symbol for symbol in symbols if "omega" in symbol.name.lower()]
    return fallback[0] if fallback else None


def as_sympy_number(value):
    value = complex(np.asarray(value).item())

    if abs(value.imag) < 1e-14:
        return sp.Float(value.real, 16)

    return sp.Float(value.real, 16) + sp.I * sp.Float(value.imag, 16)


def infer_numeric_value(symbol):
    name = symbol.name
    name_lower = name.lower()

    if symbol == Delta_sym or "delta" in name_lower:
        return as_sympy_number(DELTA_VALUE)

    if symbol == lam_sym or "lambda" in name_lower or name_lower == "rrr":
        return as_sympy_number(LAMBDA_VALUE)

    if "100" in name or "001" in name:
        return as_sympy_number(COUPLING_VALUE)

    if "E" in name:
        index_match = re.search(r"(\d+)(?!.*\d)", name)

        if index_match is not None:
            energy_index = int(index_match.group(1))

            if energy_index >= len(simulation.E_array):
                raise IndexError(f"Energieindex {energy_index} aus Symbol '{name}' ist ungültig.")

            return as_sympy_number(simulation.E_array[energy_index])

    # Entspricht dem bisherigen else-Zweig params[symbol] = 1.0
    return as_sympy_number(DEFAULT_NUMERIC_VALUE)


def canonicalize_remaining_variables(expressions):
    all_symbols = collect_free_symbols(expressions)
    canonical_map = {}

    for symbol in all_symbols:
        name_lower = symbol.name.lower()

        if symbol != t_sym and name_lower == "t":
            canonical_map[symbol] = t_sym
        elif symbol != sigma_sym and "sigma" in name_lower:
            canonical_map[symbol] = sigma_sym
        elif symbol != tg_sym and ("t_g" in name_lower or name_lower == "tg"):
            canonical_map[symbol] = tg_sym

    return tuple(sp.sympify(expression).subs(canonical_map) for expression in expressions)


def build_fixed_parameter_map(expressions, wd0_symbol):
    fixed_parameter_map = {}

    for symbol in collect_free_symbols(expressions):
        if symbol in ALLOWED_SYMBOLS or symbol == wd0_symbol:
            continue

        fixed_parameter_map[symbol] = infer_numeric_value(symbol)

    return fixed_parameter_map


def simplify_expression(expression):
    return sp.factor_terms(sp.simplify(sp.together(expression.doit())))


# ------------------------------------------------------------------------------
# 3. Vollständige symbolische Reduktion einer Pulsform
# ------------------------------------------------------------------------------

def reduce_pulse_equations(pulse_type):
    print(f"Reduziere '{pulse_type}' auf t, sigma und tg ...")
    start_time = time.perf_counter()

    _, pulse_substitutions_raw = h_target_symbolic_abstract(Delta=Delta_sym, rrr=lam_sym, tg=tg_sym, sigma_r=sigma_sym, pulse=pulse_type, t=t_sym, order=EXPANSION_ORDER)

    pulse_substitutions = {
        key: sp.sympify(value).doit()
        for key, value in pulse_substitutions_raw.items()
    }

    Ar_resolved = sol_Ar.subs(pulse_substitutions).doit()
    Ai_resolved = sol_Ai.subs(pulse_substitutions).doit()
    dwd_resolved = sol_dwd.subs(pulse_substitutions).doit()

    Ar_resolved, Ai_resolved, dwd_resolved = canonicalize_remaining_variables((Ar_resolved, Ai_resolved, dwd_resolved))

    wd0_symbol = find_wd0_symbol(dwd_resolved)
    fixed_parameter_map = build_fixed_parameter_map((Ar_resolved, Ai_resolved, dwd_resolved), wd0_symbol)

    Ar_numeric = Ar_resolved.subs(fixed_parameter_map).doit()
    Ai_numeric = Ai_resolved.subs(fixed_parameter_map).doit()
    dwd_numeric = dwd_resolved.subs(fixed_parameter_map).doit()

    wd0_solution = None

    if wd0_symbol is not None:
        boundary_equation = sp.Eq(dwd_numeric.subs(t_sym, 0), 0)
        wd0_candidates = sp.solve(boundary_equation, wd0_symbol, dict=False)

        if not wd0_candidates:
            raise ValueError(f"Für '{pulse_type}' konnte {wd0_symbol} nicht aus δω_d(0) = 0 bestimmt werden.")

        wd0_solution = simplify_expression(wd0_candidates[0])

        Ar_numeric = Ar_numeric.subs(wd0_symbol, wd0_solution)
        Ai_numeric = Ai_numeric.subs(wd0_symbol, wd0_solution)
        dwd_numeric = dwd_numeric.subs(wd0_symbol, wd0_solution)

    Ar_reduced = simplify_expression(Ar_numeric)
    Ai_reduced = simplify_expression(Ai_numeric)
    dwd_reduced = simplify_expression(dwd_numeric)

    reduced_expressions = {
        "Ar": Ar_reduced,
        "Ai": Ai_reduced,
        "dwd": dwd_reduced,
        "wd0": wd0_solution,
    }

    remaining_symbols = collect_free_symbols(expression for expression in reduced_expressions.values() if expression is not None)
    unexpected_symbols = remaining_symbols - ALLOWED_SYMBOLS

    if unexpected_symbols:
        symbol_names = ", ".join(sorted(str(symbol) for symbol in unexpected_symbols))
        raise ValueError(f"Nach der Reduktion sind noch unerwartete Symbole vorhanden: {symbol_names}")

    remaining_names = ", ".join(sorted(str(symbol) for symbol in remaining_symbols))
    elapsed_time = time.perf_counter() - start_time

    print(f"Verbleibende Symbole: {remaining_names}")
    print(f"Fertig nach {elapsed_time:.2f} s.\n")

    return reduced_expressions


# ------------------------------------------------------------------------------
# 4. Gleichungen für beide Pulsformen vorberechnen
# ------------------------------------------------------------------------------

reduced_pulse_equations = {
    "gauss": reduce_pulse_equations(pulse_type="gauss"),
    "tanh": reduce_pulse_equations(pulse_type="tanh"),
}


# Direkter Zugriff für die weitere Verarbeitung:
Ar_gauss_expr = reduced_pulse_equations["gauss"]["Ar"]
Ai_gauss_expr = reduced_pulse_equations["gauss"]["Ai"]
dwd_gauss_expr = reduced_pulse_equations["gauss"]["dwd"]

Ar_tanh_expr = reduced_pulse_equations["tanh"]["Ar"]
Ai_tanh_expr = reduced_pulse_equations["tanh"]["Ai"]
dwd_tanh_expr = reduced_pulse_equations["tanh"]["dwd"]

Reduziere 'gauss' auf t, sigma und tg ...
Verbleibende Symbole: \sigma, t, t_g
Fertig nach 0.57 s.

Reduziere 'tanh' auf t, sigma und tg ...
tanh
Verbleibende Symbole: \sigma, t, t_g
Fertig nach 1.01 s.



In [9]:

# ------------------------------------------------------------------------------
# 1. Werte, die erst jetzt eingesetzt werden
# ------------------------------------------------------------------------------

tg_val = 250.0
sigma_r_g_val = 0.2
sigma_r_t_val = 0.1
number_of_time_points = 1000


# ------------------------------------------------------------------------------
# 2. Numerische Funktionen erzeugen
# ------------------------------------------------------------------------------

numeric_pulse_functions = {
    pulse_type: {
        quantity: sp.lambdify((t_sym, sigma_sym, tg_sym), expression, modules=["numpy"])
        for quantity, expression in pulse_data.items()
        if quantity in ("Ar", "Ai", "dwd")
    }
    for pulse_type, pulse_data in reduced_pulse_equations.items()
}


def evaluate_real_function(function, t_values, sigma_value, tg_value):
    values = np.asarray(function(t_values, sigma_value, tg_value), dtype=np.complex128)

    if values.ndim == 0 or values.size == 1:
        values = np.full(t_values.shape, values.squeeze(), dtype=np.complex128)
    else:
        values = np.broadcast_to(values, t_values.shape).copy()

    if not np.all(np.isfinite(values)):
        raise FloatingPointError("Bei der numerischen Auswertung sind NaN- oder Inf-Werte aufgetreten.")

    return np.real_if_close(values).real


def evaluate_pulse(pulse_type, sigma_value, tg_value, number_of_points=1000):
    functions = numeric_pulse_functions[pulse_type]
    t_values = np.linspace(0.0, tg_value, number_of_points)

    Ar_values = evaluate_real_function(functions["Ar"], t_values, sigma_value, tg_value)
    Ai_values = evaluate_real_function(functions["Ai"], t_values, sigma_value, tg_value)
    dwd_values = evaluate_real_function(functions["dwd"], t_values, sigma_value, tg_value)

    return t_values / tg_value, Ar_values, Ai_values, dwd_values


def nonzero_maximum(values):
    maximum = float(np.max(np.abs(values)))
    return maximum if maximum > 0.0 else 1.0


# ------------------------------------------------------------------------------
# 3. Paper-Style Plot
# ------------------------------------------------------------------------------

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 14,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "lines.linewidth": 2.5,
    "figure.dpi": 600,
    "mathtext.fontset": "cm",
})

fig, axes = plt.subplots(2, 2, figsize=(15, 11))


def setup_subplot(ax, label, title):
    ax.grid(True, linestyle=":", alpha=0.7)
    ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.5)
    ax.set_xlim(0.0, 1.0)
    ax.set_xlabel(r"Time $t/t_g$")
    ax.set_title(title, pad=12)
    ax.text(0.03, 0.88, f"({label})", transform=ax.transAxes, fontsize=22, fontweight="bold", va="top")


pulse_configurations = [
    ("gauss", sigma_r_g_val, "Gaussian DRAG Envelope", "Gaussian Dynamic Shift", axes[0], ("a", "b")),
    ("tanh", sigma_r_t_val, "Tanh DRAG Envelope", "Tanh Dynamic Shift", axes[1], ("c", "d")),
]


for pulse_type, sigma_value, envelope_title, shift_title, (ax_envelope, ax_shift), (label_envelope, label_shift) in pulse_configurations:
    t_normalized, Ar_values, Ai_values, dwd_values = evaluate_pulse(pulse_type=pulse_type, sigma_value=sigma_value, tg_value=tg_val, number_of_points=number_of_time_points)

    Ar_scale = nonzero_maximum(Ar_values)
    dwd_scale = nonzero_maximum(dwd_values)

    ax_envelope.plot(t_normalized, Ar_values / Ar_scale, color="#1f4e78", label=r"$A_r(t)/\max|A_r|$")
    ax_envelope.plot(t_normalized, 10.0 * Ai_values / Ar_scale, color="#a61c1c", label=r"$10\times A_i(t)/\max|A_r|$")
    ax_envelope.set_ylabel("Normalized Envelopes")
    ax_envelope.legend(loc="upper right", frameon=True, edgecolor="black")
    setup_subplot(ax=ax_envelope, label=label_envelope, title=rf"{envelope_title} ($\sigma_r={sigma_value}$)")

    ax_shift.plot(t_normalized, dwd_values / dwd_scale, color="#127352", label=r"$\delta\omega_d(t)/\max|\delta\omega_d|$")
    ax_shift.set_ylabel("Normalized Frequency Deviation")
    ax_shift.legend(loc="upper right", frameon=True, edgecolor="black")
    setup_subplot(ax=ax_shift, label=label_shift, title=rf"{shift_title} ($\sigma_r={sigma_value}$)")


fig.tight_layout()
fig.savefig("Figure/drag_pulse_shapes.pdf", bbox_inches="tight")
plt.show()

# Now save the data

In [10]:
# ==============================================================================
# SPEICHERN DER SYMBOLISCHEN PULSFORMEN
# ==============================================================================

from pathlib import Path
import numpy as np
import sympy as sp


SAVE_DIRECTORY = Path("operational_res/analytical_drive_data")


def save_analytical_drive(pulse_type, pulse_data, save_directory=SAVE_DIRECTORY):
    save_directory = Path(save_directory)
    save_directory.mkdir(parents=True, exist_ok=True)

    file_path = save_directory / f"analytical_drive_{pulse_type}.npz"

    np.savez_compressed(
        file_path,
        pulse_type=pulse_type,
        Ar_srepr=sp.srepr(pulse_data["Ar"]),
        Ai_srepr=sp.srepr(pulse_data["Ai"]),
        dwd_srepr=sp.srepr(pulse_data["dwd"]),
        wd0_srepr=sp.srepr(pulse_data["wd0"]) if pulse_data["wd0"] is not None else "",
    )

    print(f"Symbolische Pulsform gespeichert: '{file_path}'")

    return file_path

if not symbolic:
    saved_drive_file = {
        pulse_type: save_analytical_drive(pulse_type=pulse_type, pulse_data=pulse_data)
        for pulse_type, pulse_data in reduced_pulse_equations.items()
    }

Symbolische Pulsform gespeichert: 'operational_res/analytical_drive_data/analytical_drive_gauss.npz'
Symbolische Pulsform gespeichert: 'operational_res/analytical_drive_data/analytical_drive_tanh.npz'
